# MODULE 3 - EMBEDDING AND VECTOR SIMILARITY

CORE CONCEPTS I COVERED IN THIS SECTION:  

1. What Embeddings Actually Are:

    - Numbers that capture meaning
    - The GPS analogy for understanding
    - Why computers need this representation

2. Traditional vs. Semantic Search:

    - Keyword matching (old way)
    - Meaning matching (new way)
    - Live demonstration of the difference

3. Creating Embeddings:

    - Using sentence-transformers
    - Understanding the 384-dimensional space
    - What those numbers actually represent

4. Measuring Similarity:

    - Cosine similarity explained simply
    - Why angles matter in vector space
    - Interpreting similarity scores


5. Hands-on Search Engine:

    - Build a working semantic search in ~20 lines
    - See embeddings solve real problems
    - Query: "How do plants make food?" → Finds "Photosynthesis"

In [1]:
# installing dependencies
!pip install sentence-transformers numpy scikit-learn matplotlib -q

In [2]:
# importing libraries
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from typing import List

WHAT ARE EMBEDDINGS?

mbeddings are numbers that represent the MEANING of text.

ANALOGY:
Think of GPS coordinates:
- "Eiffel Tower" → [48.8584, 2.2945]
- "Statue of Liberty" → [40.6892, -74.0445]

These numbers tell you:
1. Where something is
2. How close things are to each other

Text embeddings work the same way:
- "dog" → [0.2, 0.8, 0.1, ..., 0.5]  (384 numbers)
- "puppy" → [0.25, 0.82, 0.09, ..., 0.48]  (384 numbers)

Similar meanings → Similar numbers!

WHY IS THIS POWERFUL?

Computers can't understand "dog" and "puppy" are related.
But they CAN calculate that [0.2, 0.8, ...] is close to [0.25, 0.82, ...]

This is the foundation of semantic search!

## Understanding - Traditional VS semantic search

In [3]:
documents = [
    "The canine was barking loudly in the park",
    "A puppy played with a ball",
    "The car engine makes noise",
    "Dogs are loyal companions",
    "The feline sat on the mat"
]

query = "dog behavior"

print(f"Query: '{query}'")
print(f"Documents:")

for i, doc in enumerate(documents, 1):
    print(f"  {i}. {doc}")

print("\n" + "-" * 70)
print("TRADITIONAL KEYWORD SEARCH:")
print("-" * 70)

print("""
Looks for exact word matches: "dog"

Results:
  4. Dogs are loyal companions  ← Only this matches!

Misses:
  1. "canine" (means dog!)
  2. "puppy" (means young dog!)

PROBLEM: Relies on exact words, not meaning
""")

Query: 'dog behavior'
Documents:
  1. The canine was barking loudly in the park
  2. A puppy played with a ball
  3. The car engine makes noise
  4. Dogs are loyal companions
  5. The feline sat on the mat

----------------------------------------------------------------------
TRADITIONAL KEYWORD SEARCH:
----------------------------------------------------------------------

Looks for exact word matches: "dog"

Results:
  4. Dogs are loyal companions  ← Only this matches!

Misses:
  1. "canine" (means dog!)
  2. "puppy" (means young dog!)

PROBLEM: Relies on exact words, not meaning



## Semantic search - Understands MEANING, not just words!


---


Results (by relevance):

  4. Dogs are loyal companions      ← Direct match
  1. The canine was barking...      ← "canine" = dog
  2. A puppy played...              ← "puppy" = young dog
  
Finds relevant docs even with different words!

In [4]:
# creating first embeddings
# loading a pretrained emebedding model - all-MiniLM-L6_v2 (small, fast, good quality)
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# encoding some sample text for demo
sample_texts = [
    "The dog is running",
    "A puppy is playing",
    "The cat is sleeping"
]

embeddings = model.encode(sample_texts)
print(f"Number of texts: {len(sample_texts)}")
print(f"Embedding shape: {embeddings.shape}")
print(f"Meaning: {embeddings.shape[0]} vectors, each with {embeddings.shape[1]} dimensions")

Number of texts: 3
Embedding shape: (3, 384)
Meaning: 3 vectors, each with 384 dimensions


In [6]:
print(f"Embedding (first 10 dimensions): {embeddings[0][:10]}")
print(f"Embedding (last 10 dimensions): {embeddings[0][-10:]}")

Embedding (first 10 dimensions): [ 0.00804122  0.00034923  0.01742687  0.05856586  0.00660967  0.01359686
 -0.1069337  -0.0176941   0.03246238 -0.02650031]
Embedding (last 10 dimensions): [ 0.02005637  0.07717216 -0.03738118 -0.07103181 -0.01076988  0.03651291
 -0.01985378  0.01479283  0.06117813  0.07585194]


KEY OBSERVATIONS:
1. Each text becomes a vector of 384 numbers
2. Numbers are between -1 and 1 (normalized)
3. These numbers capture the MEANING of the text
4. You can't read meaning from individual numbers

In [7]:
# let's measure similarity - cosine similarity
# cosine_similarity = (A · B) / (||A|| × ||B||)
similarities = cosine_similarity([embeddings[0]], embeddings)[0]

for i, text in enumerate(sample_texts):
  similarity = similarities[i]
  print("Comparing:")
  print(f"{sample_texts[0]}")
  print(f"{text}")
  print(f"  → Similarity: {similarity:.4f} {'⭐' * int(similarity * 5)}")

Comparing:
The dog is running
The dog is running
  → Similarity: 1.0000 ⭐⭐⭐⭐
Comparing:
The dog is running
A puppy is playing
  → Similarity: 0.5625 ⭐⭐
Comparing:
The dog is running
The cat is sleeping
  → Similarity: 0.1410 


## Visualizing the embeddings

Embeddings have 384 dimensions - impossible to visualize!
Let's use PCA to compress to 2D ( for visualizing only )

there will be some information loss but helps use see the concept.

In [8]:
from sklearn.decomposition import PCA

varied_texts = [
    # Animal-related
    "The dog is barking",
    "A puppy is playing",
    "The cat is meowing",
    # Vehicle-related
    "The car is fast",
    "A truck is heavy",
    # Food-related
    "Pizza is delicious",
    "Pasta tastes great",
]

# encoding texts
varied_embeddings = model.encode(varied_texts)

In [9]:
# reducing to 2d
pca = PCA(n_components=2)
reduced_embeddings = pca.fit_transform(varied_embeddings)

In [10]:
for text, coords in zip(varied_texts, reduced_embeddings):
    print(f"  '{text[:30]}...' → [{coords[0]:6.3f}, {coords[1]:6.3f}]")

  'The dog is barking...' → [-0.500, -0.338]
  'A puppy is playing...' → [-0.513, -0.208]
  'The cat is meowing...' → [-0.440, -0.035]
  'The car is fast...' → [ 0.140,  0.616]
  'A truck is heavy...' → [ 0.107,  0.722]
  'Pizza is delicious...' → [ 0.588, -0.426]
  'Pasta tastes great...' → [ 0.618, -0.330]


Similar meaning cluster together in embedding space.

## Semantic Search in Action

In [11]:
# example document collection
documents = [
    "Machine learning models can predict future trends",
    "Neural networks are inspired by the human brain",
    "The Eiffel Tower is located in Paris, France",
    "Deep learning requires large amounts of data",
    "Python is a popular programming language",
    "The Amazon rainforest is the largest tropical rainforest",
    "Transformers revolutionized natural language processing",
    "The Great Wall of China is visible from space",
]

In [12]:
# encode all documents
doc_embeddings = model.encode(documents)
doc_embeddings

array([[-0.03622404, -0.07102228,  0.05316397, ..., -0.04548401,
        -0.01575573,  0.00584953],
       [-0.06021561, -0.07078337,  0.06588805, ...,  0.14998831,
         0.04440377, -0.05624178],
       [ 0.04477228,  0.05103539,  0.01053969, ...,  0.04488286,
         0.07727335,  0.04172751],
       ...,
       [ 0.1243536 ,  0.00224038, -0.01846378, ..., -0.02570201,
         0.01970478,  0.0621177 ],
       [-0.03012687, -0.00693713,  0.04525302, ...,  0.01102001,
         0.0486651 , -0.00727452],
       [ 0.03942326,  0.07330817,  0.05258238, ...,  0.015     ,
        -0.09715851,  0.06561728]], dtype=float32)

In [13]:
# user query
query = "artificial intelligence and neural nets"
# creating query embedding
query_embedding = model.encode([query])
# fine similarity with documents
similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

In [14]:
# ranking documents and sorting
results = list(zip(documents, similarities))
results.sort(key=lambda x: x[1], reverse=True)

for rank, (doc, score) in enumerate(results, 1):
  relevance_bar = "█" * int(score * 20)
  print(f"\n{rank}. [{score:.4f}] {relevance_bar}")
  print(f"   {doc}")


1. [0.5354] ██████████
   Neural networks are inspired by the human brain

2. [0.3555] ███████
   Machine learning models can predict future trends

3. [0.2546] █████
   Deep learning requires large amounts of data

4. [0.1592] ███
   Transformers revolutionized natural language processing

5. [0.1238] ██
   Python is a popular programming language

6. [0.0436] 
   The Eiffel Tower is located in Paris, France

7. [0.0296] 
   The Great Wall of China is visible from space

8. [-0.0281] 
   The Amazon rainforest is the largest tropical rainforest


Understanding Embedding dimensions-

WHAT ARE DIMENSIONS?

Think of dimensions as "features" the model learned:

Dimensions 1 might capture: "Is this about animals?"

Dimension 2 might capture: "Is this formal or casual?"

Dimension 3 might capture: "Is this about technology?"
... and so on for all 384 dimensions!

COMMON EMBEDDING SIZES:

- all-MiniLM-L6-v2: 384 dimensions (we're using this)
- BERT-base: 768 dimensions
- OpenAI ada-002: 1536 dimensions
- Sentence-BERT large: 1024 dimensions

MORE DIMENSIONS = BETTER?
Not always!

✅ Pros of more dimensions:
  - Can capture more nuanced meanings
  - Better at complex relationships
  
❌ Cons of more dimensions:
  - Slower to compute
  - Requires more storage
  - Risk of overfitting on small datasets

FOR RAG: 384-768 dimensions is usually perfect!

COMPARING EMBEDDING MODELS

Different models are trained differently and excel at different tasks:

1. all-MiniLM-L6-v2 (we're using this)

   ✅ Fast and small (80MB)
   
   ✅ Good for general semantic similarity
   
   ✅ Great for learning and prototyping
   
2. all-mpnet-base-v2
   
   ✅ Better quality, slower
   
   ✅ 420MB model
   
   ✅ Better for production

3. multi-qa-MiniLM-L6-cos-v1
   ✅ Optimized for question-answering
   ✅ Best when queries are questions
   
4. OpenAI text-embedding-ada-002

   ✅ Very high quality

   ❌ Requires API calls (costs money)
   
   ❌ Not local/private

PRACTICAL TIPS FOR EMBEDDINGS

1. Chunk size matters
    - DO: chunk before embedding
    - DON'T: embed entire documents
  Embedding captures meaning best with 1-2 paragraphs, not whole books.

2. Same model for everything
    - DO: user same model for documents and queries
    - DON'T: mix different embedding models
    Each model has their own embedding space. They're not compatible.

3. Cache embeddings
    - DO: save document embeddings ( they don't change )
    - DON'T: re-compute embeddings every time.
    Embeddings are slow. Do it once and save it.

4. Normalize embeddings
    - DO: user normalized embeddings ( length =1 )
    - DON'T: use raw outputs
    Makes cosine similarity faster ( most models handles automatically )

5. Test with your data
    - DO: Try 2-3 models on your specific use case
    - DON'T: assume one model fits all
    Different domains have different characterstics.

## Hands-on Exercise

In [15]:
# let's build a mini search engine
def simple_search_engine(documents: List[str], query:str, top_k:int=3):
  """ simple sementice search engine
  returns: top-k relevant documents with scores
  """
  # encode all documents
  doc_embeddings = model.encode(documents, show_progress_bar = False)
  #encode the query
  query_embedding = model.encode([query], show_progress_bar=False)
  # calculate similarities
  similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
  # get top k results
  top_indices = np.argsort(similarities)[::-1][:top_k]

  results = []
  for idx in top_indices:
    results.append({
        'document': documents[idx],
        'score': similarities[idx],
        'index':idx}
    )
  return results


In [16]:
# testing the search engine
knowledge_base = [
    "Paris is the capital of France and home to the Eiffel Tower",
    "The Python programming language is known for its simplicity",
    "Machine learning is a subset of artificial intelligence",
    "The Great Barrier Reef is the world's largest coral reef system",
    "Shakespeare wrote Romeo and Juliet in the 16th century",
    "Photosynthesis is how plants convert sunlight into energy",
    "The Internet was invented in the late 1960s",
    "Mount Everest is the tallest mountain on Earth",
]

test_query = "How do plants make food?"
results = simple_search_engine(knowledge_base, test_query, top_k=3)

for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result['score']:.4f}")
    print(f"   {result['document']}")


1. Score: 0.4988
   Photosynthesis is how plants convert sunlight into energy

2. Score: 0.0792
   The Python programming language is known for its simplicity

3. Score: 0.0507
   The Internet was invented in the late 1960s


Different words but same meaning. The search found 'Photosynthesis' even though the query used 'plants make food'.

1. WHAT ARE EMBEDDINGS?

   ✅ Numbers that represent meaning

   ✅ Similar meanings = Similar numbers

   ✅ Enable semantic search (meaning-based, not word-based)

2. HOW DO THEY WORK?

   ✅ Text → Model → Vector (e.g., 384 dimensions)

   ✅ Each dimension captures a learned "feature"

   ✅ The pattern of all dimensions together = meaning

3. MEASURING SIMILARITY:

   ✅ Cosine similarity is the standard metric

   ✅ Range: -1 to 1 (higher = more similar)

   ✅ Fast to compute, works well in practice

4. CHOOSING A MODEL:

   ✅ all-MiniLM-L6-v2: Fast, good for learning

   ✅ all-mpnet-base-v2: Better quality

   ✅ Use same model for docs and queries!

5. WHY THIS MATTERS FOR RAG:

   ✅ RAG retrieval is just semantic search!

   ✅ User query → Find similar chunks → Send to LLM

   ✅ Better embeddings = Better retrieval = Better answers